# Shioaji SDK 快速入門

這個 notebook 展示如何使用 Shioaji SDK 進行基本的股票交易操作。

## 前置準備

1. 安裝套件：`pip install shioaji-mcp`
2. 設定環境變數或在程式碼中提供 API 金鑰

## 1. 初始化 SDK

In [ ]:
from shioaji_mcp.sdk import ShioajiSDK
import os

# 方法 1: 使用環境變數（建議）
# export SHIOAJI_API_KEY=your_key
# export SHIOAJI_SECRET_KEY=your_secret
sdk = ShioajiSDK()

# 方法 2: 直接提供金鑰（不建議在生產環境使用）
# sdk = ShioajiSDK(
#     api_key="your_api_key",
#     secret_key="your_secret_key"
# )

print("✅ Connected to Shioaji!")

## 2. 查詢帳戶資訊

In [ ]:
import pandas as pd

# 取得帳戶列表
accounts = sdk.get_accounts()

# 顯示為表格
df_accounts = pd.DataFrame(accounts)
df_accounts

## 3. 搜尋股票

In [ ]:
# 搜尋台積電
results = sdk.search_stocks("台積電", limit=5)
df_search = pd.DataFrame(results)
df_search

## 4. 取得即時報價

In [ ]:
# 單一股票報價
quote = sdk.get_quote("2330")  # 台積電

print(f"公司: {quote['name']}")
print(f"代碼: {quote['code']}")
print(f"價格: ${quote['close']:,.2f}")
print(f"漲跌: {quote['close'] - quote['open']:+,.2f}")
print(f"成交量: {quote['volume']:,}")
print(f"最高: ${quote['high']:,.2f}")
print(f"最低: ${quote['low']:,.2f}")

In [ ]:
# 多支股票報價
symbols = ["2330", "2317", "2454"]  # 台積電、鴻海、聯發科
quotes = sdk.get_quotes(symbols)

df_quotes = pd.DataFrame(quotes)
df_quotes['change'] = df_quotes['close'] - df_quotes['open']
df_quotes['change_pct'] = (df_quotes['change'] / df_quotes['open'] * 100).round(2)
df_quotes[['code', 'name', 'close', 'change', 'change_pct', 'volume']]

## 5. 取得歷史資料

In [ ]:
# 取得最近 30 天的 K 線資料
history = sdk.get_history("2330", days=30)

df_history = pd.DataFrame(history)
df_history['date'] = pd.to_datetime(df_history['date'])
df_history.set_index('date', inplace=True)

# 顯示最後 10 筆
df_history.tail(10)

In [ ]:
# 繪製 K 線圖
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})

# 價格圖
ax1.plot(df_history.index, df_history['close'], label='Close', linewidth=2)
ax1.fill_between(df_history.index, df_history['low'], df_history['high'], alpha=0.3)
ax1.set_title('台積電 (2330) - 收盤價', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (TWD)', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 成交量圖
ax2.bar(df_history.index, df_history['volume'], alpha=0.6)
ax2.set_title('成交量', fontsize=12)
ax2.set_ylabel('Volume', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 查詢持倉

In [ ]:
# 取得所有持倉
positions = sdk.get_positions()

# 股票持倉
if positions.get('stock'):
    print("📈 股票持倉:")
    df_stock_pos = pd.DataFrame(positions['stock'])
    display(df_stock_pos)
else:
    print("📈 無股票持倉")

# 期貨持倉
if positions.get('futures'):
    print("\n📊 期貨持倉:")
    df_futures_pos = pd.DataFrame(positions['futures'])
    display(df_futures_pos)
else:
    print("\n📊 無期貨持倉")

## 7. 查詢帳戶餘額

In [ ]:
# 取得帳戶餘額
balance = sdk.get_balance()

# 股票帳戶
if 'stock' in balance and 'error' not in balance['stock']:
    print("💰 股票帳戶:")
    stock_bal = balance['stock']
    print(f"  可用餘額: ${stock_bal['available']:,.2f}")
    print(f"  購買力: ${stock_bal['buying_power']:,.2f}")
    print(f"  持股市值: ${stock_bal['stock_value']:,.2f}")
    print(f"  總資產: ${stock_bal['total']:,.2f}")

# 期貨帳戶
if 'futures' in balance and 'error' not in balance['futures']:
    print("\n💰 期貨帳戶:")
    fut_bal = balance['futures']
    print(f"  可用餘額: ${fut_bal['available']:,.2f}")
    print(f"  保證金使用: ${fut_bal['margin_used']:,.2f}")
    print(f"  原始保證金: ${fut_bal['initial_margin']:,.2f}")
    print(f"  維持保證金: ${fut_bal['maintenance_margin']:,.2f}")

## 8. 技術分析範例

使用歷史資料計算簡單的技術指標

In [ ]:
# 計算移動平均線
df_history['MA5'] = df_history['close'].rolling(window=5).mean()
df_history['MA10'] = df_history['close'].rolling(window=10).mean()
df_history['MA20'] = df_history['close'].rolling(window=20).mean()

# 繪圖
plt.figure(figsize=(12, 6))
plt.plot(df_history.index, df_history['close'], label='Close', linewidth=2)
plt.plot(df_history.index, df_history['MA5'], label='MA5', linewidth=1.5, linestyle='--')
plt.plot(df_history.index, df_history['MA10'], label='MA10', linewidth=1.5, linestyle='--')
plt.plot(df_history.index, df_history['MA20'], label='MA20', linewidth=1.5, linestyle='--')

plt.title('台積電 (2330) - 移動平均線', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (TWD)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 斷開連線

In [ ]:
# 使用完畢後斷開連線
sdk.disconnect()
print("✅ Disconnected from Shioaji")

## 進階使用

### 下單功能

⚠️ **注意**: 下單功能需要設定 `SHIOAJI_TRADING_ENABLED=true` 環境變數

```python
# 市價單
order = sdk.place_order("2330", "Buy", 1000)

# 限價單
order = sdk.place_order("2330", "Buy", 1000, price=500.0)

# 不同的訂單類型
order = sdk.place_order("2330", "Buy", 1000, price=500.0, order_type="IOC")
```

### 取得原始 Shioaji API

如果需要使用 Shioaji 的進階功能，可以取得原始 API 物件：

```python
raw_api = sdk.api
# 使用原始 Shioaji API 的所有功能
```

### 更多資源

- [Shioaji 官方文件](https://sinotrade.github.io/)
- [專案 GitHub](https://github.com/musingfox/shioaji-mcp)
- [永豐金證券 API](https://www.sinotrade.com.tw/)